# 조합3(11개 변수) 데이터 누수 없는 재생성

## 배경

기존 "조합3" 11개 변수 중 아래 5개는 실제로 만든 코드/버전이 여러 개 섞여 있어서
데이터 누수 여부가 불확실했다.

1. `recent_24h_high_amt_count`
2. `amt_zscore_card` (+ 내부적으로 쓰는 `customer_mean_amt`, `customer_std_amt`)
3. `amt_to_prior_median_ratio`
4. `prior_normal_median_amt`
5. `rolling_sum_amt_1h`

특히 `02_train_preprocessing_eda.ipynb`에서 확인된 "정상 거래 기준 고객별 평소
거래금액"은 train 전체(과거+미래 거래 전부)를 `groupby.median()`으로 계산한 뒤
모든 행에 붙이는 방식이었다. 이건 이른 시점 거래에 미래 정보가 섞이는 명백한
누수였다. 이 노트북은 그 문제를 고쳐서, "그 시점 이전 데이터만" 사용하도록
5개 변수를 전부 다시 만든다.

문제없다고 이미 확인된 6개 변수(`category`, `amt`, `trans_hour`, `age`,
`count_30min`, `high_speed`)는 팀이 이미 검증한 로직(`final_train_code.ipynb`)을
그대로 재사용한다.

## 핵심 원칙 (시점 인과성)

- 모든 파생변수는 "그 거래 시점에 이미 알 수 있었던 정보"만 사용한다.
- `amt`, 거래 위치, 거래 시각은 거래 발생 즉시 알 수 있다고 가정한다.
- `is_fraud`(정답)는 그 거래 자신에게는 물론, "그 거래 이후" 어떤 계산에도 쓰지
  않는다. `prior_normal_median_amt`처럼 과거의 `is_fraud`를 쓰는 변수도, 반드시
  "현재 거래보다 시간상 앞선" 거래의 `is_fraud`만 사용한다.
- Train -> Test 경계를 넘어갈 때도 원칙은 같다. amt/위치/시각처럼 실시간으로
  알 수 있는 정보 기반 변수(고객 평균금액, 최근 24시간 카운트, 최근 1시간
  누적금액, 30분 반복횟수, 이동속도)는 Train에서 이어진 상태를 Test에서도
  계속 갱신한다. 반면 정상거래 판정에 `is_fraud`가 필요한 `prior_normal_median_amt`는,
  실무에서 라벨 확정이 즉시 되지 않는다고 보수적으로 가정하여 Train 종료 시점의
  카드별 값에 "고정"하고 Test 구간에서는 갱신하지 않는다. (팀이 이미 작성한
  `test_dataset_reproduction/step4c` 스크립트와 동일한 설계.)

## 사용법

1. 아래 "0. 경로 설정" 셀의 `CANDIDATE_TRAIN_PATHS` / `CANDIDATE_TEST_PATHS`에
   로컬 raw csv 경로가 이미 들어있다. 다르면 직접 수정해도 된다.
2. 위에서부터 순서대로 셀을 실행한다 (Run All).
3. 결과물:
   - `train_11features.csv` (fraudTrain.csv 기반)
   - `test_11features.csv`  (fraudTest.csv 기반)
   - 출력 컬럼: `trans_num, trans_date_trans_time, cc_num, category, amt, trans_hour,
     age, recent_24h_high_amt_count, amt_to_prior_median_ratio, rolling_sum_amt_1h,
     amt_zscore_card, prior_normal_median_amt, count_30min, high_speed, is_fraud`
     (앞의 세 개는 식별/디버깅용 키이고, 모델 입력은 `category` ~ `high_speed`
     11개 + `is_fraud` 타깃이다.)
4. 마지막 "검증" 섹션에서 팀이 끝내지 못했던 "재계산 검증"을 자동으로 수행한다
   (`recent_24h_high_amt_count`, `amt_zscore_card`, `prior_normal_median_amt` 각각
   임의 고객 표본에 대해 독립적인 방법으로 재계산해서 비교).

## 0. 경로 설정 및 공통 상수

In [11]:
from __future__ import annotations

import heapq
from pathlib import Path

import numpy as np
import pandas as pd

CANDIDATE_TRAIN_PATHS = [
    Path(r"C:\Users\splen\OneDrive\Desktop\BDAI_\BOOSTMAP\Fraud-FDS-Project\data\raw\fraudTrain.csv"),
    Path(r"C:\Users\splen\OneDrive\Desktop\BDAI\BOOSTMAP\Fraud-FDS-Project\data\raw\fraudTrain.csv"),
    Path("data/raw/fraudTrain.csv"),
    Path("fraudTrain.csv"),
]

CANDIDATE_TEST_PATHS = [
    Path(r"C:\Users\splen\OneDrive\Desktop\BDAI_\BOOSTMAP\Fraud-FDS-Project\data\raw\fraudTest.csv"),
    Path(r"C:\Users\splen\OneDrive\Desktop\BDAI\BOOSTMAP\Fraud-FDS-Project\data\raw\fraudTest.csv"),
    Path("data/raw/fraudTest.csv"),
    Path("fraudTest.csv"),
]

OUT_TRAIN = Path("train_11features.csv")
OUT_TEST = Path("test_11features.csv")

ONLINE_CATEGORIES = {"shopping_net", "misc_net", "grocery_net"}
HIGH_AMT_THRESHOLD = 500
HIGH_SPEED_THRESHOLD_KMH = 100


def find_path(candidates: list[Path]) -> Path:
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(
        "원본 csv를 찾지 못했습니다. 이 셀의 "
        "CANDIDATE_TRAIN_PATHS / CANDIDATE_TEST_PATHS 를 실제 경로로 "
        "고쳐주세요.\n시도한 경로:\n" + "\n".join(str(p) for p in candidates)
    )


## 1. 유틸 함수

In [12]:
def haversine(lat1, lon1, lat2, lon2):
    r = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlambda / 2) ** 2
    return 2 * r * np.arcsin(np.sqrt(a))


def exact_age(trans_dt: pd.Series, birth_dt: pd.Series) -> pd.Series:
    years = trans_dt.dt.year - birth_dt.dt.year
    not_yet = (trans_dt.dt.month < birth_dt.dt.month) | (
        (trans_dt.dt.month == birth_dt.dt.month) & (trans_dt.dt.day < birth_dt.dt.day)
    )
    return (years - not_yet.astype("int64")).astype("int64")


def load_raw(path: Path, source: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")], errors="ignore")
    required = {"trans_num", "trans_date_trans_time", "cc_num", "dob", "category", "amt",
                "merch_lat", "merch_long", "is_fraud"}
    missing = required.difference(df.columns)
    assert not missing, f"{path} 에 필수 컬럼이 없습니다: {missing}"
    assert df["trans_num"].is_unique, f"{path}: trans_num 중복 존재"
    df["_source"] = source
    return df


## 2. 안전하다고 이미 확인된 6개 변수
`category`, `amt`, `trans_hour`, `age`, `count_30min`, `high_speed`

In [13]:
def add_current_row_features(df: pd.DataFrame) -> pd.DataFrame:
    """현재 거래 자체 정보만 쓰는 변수 -> 누수 불가능."""
    df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"], errors="raise")
    df["dob"] = pd.to_datetime(df["dob"], errors="raise")
    df["trans_hour"] = df["trans_date_trans_time"].dt.hour.astype("int64")
    df["age"] = exact_age(df["trans_date_trans_time"], df["dob"])
    df["is_online"] = df["category"].isin(ONLINE_CATEGORIES)
    df["is_high_amt"] = (df["amt"] >= HIGH_AMT_THRESHOLD).astype("int64")
    return df


In [14]:
def add_offline_speed_and_count30(df: pd.DataFrame) -> pd.DataFrame:
    """
    count_30min, high_speed.
    오프라인 거래만 대상으로 하고, '직전 오프라인 거래'만 참조한다(shift(1) 계열).
    Train->Test 경계를 넘어서도 cc_num별로 이어서 계산되도록, 반드시 train+test를
    합친 데이터프레임을 시간순으로 정렬한 뒤 호출해야 한다.
    """
    df = df.sort_values(["cc_num", "trans_date_trans_time", "trans_num"], kind="mergesort").reset_index(drop=True)

    offline_mask = ~df["is_online"]
    offline_df = df.loc[offline_mask].copy()
    offline_df = offline_df.sort_values(["cc_num", "trans_date_trans_time", "trans_num"], kind="mergesort")

    # --- count_30min: 직전 30분 이내 오프라인 거래 수(현재 거래 포함) ---
    offline_df["count_30min"] = 0
    for _, idx in offline_df.groupby("cc_num", sort=False).groups.items():
        times = offline_df.loc[idx, "trans_date_trans_time"].values.astype("datetime64[s]")
        left = np.searchsorted(times, times - np.timedelta64(30, "m"))
        right = np.arange(len(times))
        offline_df.loc[idx, "count_30min"] = right - left + 1

    # --- high_speed: 직전 오프라인 거래 대비 이동속도 100km/h 이상이면 1 ---
    offline_df["prev_merch_lat"] = offline_df.groupby("cc_num")["merch_lat"].shift(1)
    offline_df["prev_merch_long"] = offline_df.groupby("cc_num")["merch_long"].shift(1)
    offline_df["prev_trans_time"] = offline_df.groupby("cc_num")["trans_date_trans_time"].shift(1)

    dist_km = haversine(
        offline_df["prev_merch_lat"], offline_df["prev_merch_long"],
        offline_df["merch_lat"], offline_df["merch_long"],
    )
    hours = (
        offline_df["trans_date_trans_time"] - offline_df["prev_trans_time"]
    ).dt.total_seconds() / 3600

    with np.errstate(divide="ignore", invalid="ignore"):
        raw_speed = np.where(hours > 0, dist_km / hours, 0.0)
    raw_speed = np.nan_to_num(raw_speed, nan=0.0, posinf=0.0, neginf=0.0)

    offline_df["high_speed"] = (raw_speed >= HIGH_SPEED_THRESHOLD_KMH).astype("int64")

    keep = offline_df[["trans_num", "count_30min", "high_speed"]]
    df = df.merge(keep, on="trans_num", how="left")
    df["count_30min"] = df["count_30min"].fillna(0).astype("int64")
    df["high_speed"] = df["high_speed"].fillna(0).astype("int64")
    return df


## 3. 재검증 대상이었던 5개 변수 (누수 없이 재계산)

In [15]:
def add_recent_24h_high_amt_count(df: pd.DataFrame) -> pd.DataFrame:
    """
    최근 24시간 고액거래(amt>=500) 횟수. closed='left'로 현재 거래 자신은 제외.
    Train->Test 전체를 합친 데이터프레임 하나에 대해 계산하므로 경계도 자연히 이어진다.
    """
    tmp = df[["cc_num", "trans_date_trans_time", "is_high_amt"]].copy()
    tmp = tmp.set_index("trans_date_trans_time")
    tmp = tmp.sort_values(["cc_num"], kind="mergesort")
    values = (
        tmp.groupby("cc_num")["is_high_amt"]
        .rolling("24h", closed="left", min_periods=0)
        .sum()
        .values
    )
    tmp2 = df.sort_values(["cc_num"], kind="mergesort")[["trans_num"]].copy()
    tmp2["recent_24h_high_amt_count"] = values
    df = df.merge(tmp2, on="trans_num", how="left")
    df["recent_24h_high_amt_count"] = df["recent_24h_high_amt_count"].fillna(0).astype("int64")
    return df


def add_rolling_sum_amt_1h(df: pd.DataFrame) -> pd.DataFrame:
    """
    최근 1시간 누적 결제금액. 기본(closed='right')이라 현재 거래를 포함하고,
    현재 시각 기준 과거 1시간 이내 거래 금액을 전부 더한다 (미래 미포함).
    """
    tmp = df[["cc_num", "trans_date_trans_time", "amt"]].copy()
    tmp = tmp.set_index("trans_date_trans_time").sort_values(["cc_num"], kind="mergesort")
    values = tmp.groupby("cc_num")["amt"].rolling("1h", min_periods=1).sum().values

    tmp2 = df.sort_values(["cc_num"], kind="mergesort")[["trans_num"]].copy()
    tmp2["rolling_sum_amt_1h"] = values
    df = df.merge(tmp2, on="trans_num", how="left")
    return df


In [16]:
def add_customer_amt_stats(df: pd.DataFrame) -> pd.DataFrame:
    """
    customer_mean_amt / customer_std_amt / amt_zscore_card.
    shift(1) + expanding: 현재 거래를 제외한, 그 이전 거래들만으로 계산.
    """
    df = df.sort_values(["cc_num", "trans_date_trans_time", "trans_num"], kind="mergesort").reset_index(drop=True)

    grouped = df.groupby("cc_num")["amt"]
    df["customer_mean_amt"] = grouped.apply(lambda s: s.shift(1).expanding().mean()).reset_index(level=0, drop=True)
    df["customer_std_amt"] = grouped.apply(lambda s: s.shift(1).expanding().std()).reset_index(level=0, drop=True)

    df["customer_mean_amt"] = df["customer_mean_amt"].fillna(0.0)
    df["customer_std_amt"] = df["customer_std_amt"].fillna(0.0)

    with np.errstate(divide="ignore", invalid="ignore"):
        z = (df["amt"] - df["customer_mean_amt"]) / df["customer_std_amt"]
    df["amt_zscore_card"] = z.replace([np.inf, -np.inf], 0.0).fillna(0.0)
    return df


In [17]:
class RunningMedian:
    """카드 한 장의 '정상 거래 금액' 중앙값을 온라인으로 유지하는 두 힙 구조."""

    __slots__ = ("lo", "hi")

    def __init__(self):
        self.lo: list[float] = []   # max-heap (음수로 저장)
        self.hi: list[float] = []   # min-heap

    def add(self, x: float) -> None:
        if not self.lo or x <= -self.lo[0]:
            heapq.heappush(self.lo, -x)
        else:
            heapq.heappush(self.hi, x)

        if len(self.lo) > len(self.hi) + 1:
            heapq.heappush(self.hi, -heapq.heappop(self.lo))
        elif len(self.hi) > len(self.lo):
            heapq.heappush(self.lo, -heapq.heappop(self.hi))

    def median(self):
        if not self.lo and not self.hi:
            return np.nan
        if len(self.lo) > len(self.hi):
            return float(-self.lo[0])
        return (float(-self.lo[0]) + float(self.hi[0])) / 2.0


In [18]:
def add_prior_normal_median_train(train_df: pd.DataFrame):
    """
    Train 구간 전용: 그 거래 '이전'의 정상거래(is_fraud==0)만으로 카드별 중앙값을
    시점별로 갱신하며 계산한다 (02_train_preprocessing_eda.ipynb에서 확인된,
    train 전체를 한 번에 groupby.median() 하던 방식의 누수를 여기서 고친다).
    """
    train_df = train_df.sort_values(
        ["cc_num", "trans_date_trans_time", "trans_num"], kind="mergesort"
    ).reset_index(drop=True)

    n = len(train_df)
    out = np.full(n, np.nan, dtype="float64")
    medians: dict = {}

    cc_arr = train_df["cc_num"].to_numpy()
    amt_arr = train_df["amt"].to_numpy()
    fraud_arr = train_df["is_fraud"].to_numpy()

    for i in range(n):
        cc = cc_arr[i]
        rm = medians.get(cc)
        out[i] = rm.median() if rm is not None else np.nan
        if fraud_arr[i] == 0:
            if rm is None:
                rm = RunningMedian()
                medians[cc] = rm
            rm.add(amt_arr[i])

    train_df["prior_normal_median_amt"] = out

    # Train 종료 시점의 카드별 최종 중앙값 스냅샷 (Test에 고정해서 쓸 값)
    final_median_by_card = {cc: rm.median() for cc, rm in medians.items()}
    return train_df, final_median_by_card


def add_prior_normal_median_test(test_df: pd.DataFrame, final_median_by_card: dict) -> pd.DataFrame:
    """
    Test 구간: Train 종료 시점에 고정된 카드별 중앙값을 그대로 사용한다(갱신 없음).
    Test 자신의 is_fraud는 절대 사용하지 않는다. Train에 없던 신규 카드는 NaN.
    """
    test_df = test_df.copy()
    test_df["prior_normal_median_amt"] = test_df["cc_num"].map(final_median_by_card)
    return test_df


def finalize_amt_ratio(df: pd.DataFrame) -> pd.DataFrame:
    with np.errstate(divide="ignore", invalid="ignore"):
        ratio = df["amt"] / df["prior_normal_median_amt"]
    df["amt_to_prior_median_ratio"] = ratio.replace([np.inf, -np.inf], np.nan)
    return df


## 4. 파이프라인 조립

In [19]:
FINAL_COLUMNS = [
    "trans_num", "trans_date_trans_time", "cc_num",
    "category", "amt", "trans_hour", "age",
    "recent_24h_high_amt_count", "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h", "amt_zscore_card", "prior_normal_median_amt",
    "count_30min", "high_speed",
    "is_fraud",
]


def build_datasets(train_path: Path, test_path: Path):
    train_raw = load_raw(train_path, "train")
    test_raw = load_raw(test_path, "test")

    combined = pd.concat([train_raw, test_raw], ignore_index=True)
    combined = add_current_row_features(combined)

    # amt/위치/시각 기반 -> train/test 경계 상관없이 이어서 계산
    combined = add_offline_speed_and_count30(combined)
    combined = add_recent_24h_high_amt_count(combined)
    combined = add_rolling_sum_amt_1h(combined)
    combined = add_customer_amt_stats(combined)

    train_df = combined.loc[combined["_source"] == "train"].copy()
    test_df = combined.loc[combined["_source"] == "test"].copy()

    # is_fraud가 필요한 prior_normal_median_amt는 train/test를 분리해서 처리
    train_df, final_median_by_card = add_prior_normal_median_train(train_df)
    test_df = add_prior_normal_median_test(test_df, final_median_by_card)

    train_df = finalize_amt_ratio(train_df)
    test_df = finalize_amt_ratio(test_df)

    # 원래 파일 내 순서(trans_date_trans_time 순)로 복원
    train_df = train_df.sort_values(["trans_date_trans_time", "trans_num"], kind="mergesort").reset_index(drop=True)
    test_df = test_df.sort_values(["trans_date_trans_time", "trans_num"], kind="mergesort").reset_index(drop=True)

    return train_df[FINAL_COLUMNS], test_df[FINAL_COLUMNS]


## 5. 검증 (팀이 끝내지 못했던 재계산 검증을 여기서 마무리)

In [20]:
def verify_recent_24h(combined_raw: pd.DataFrame, result_df: pd.DataFrame, n_sample: int = 5) -> None:
    rng = np.random.default_rng(42)
    sample_cards = rng.choice(result_df["cc_num"].unique(), size=min(n_sample, result_df["cc_num"].nunique()), replace=False)
    for cc in sample_cards:
        sub = result_df.loc[result_df["cc_num"] == cc].sort_values("trans_date_trans_time")
        raw_sub = combined_raw.loc[combined_raw["cc_num"] == cc].copy()
        raw_sub["trans_date_trans_time"] = pd.to_datetime(raw_sub["trans_date_trans_time"])
        raw_sub["is_high_amt"] = (raw_sub["amt"] >= HIGH_AMT_THRESHOLD).astype(int)
        raw_sub = raw_sub.sort_values("trans_date_trans_time")
        times = raw_sub["trans_date_trans_time"].to_numpy()
        for _, row in sub.head(20).iterrows():
            t = np.datetime64(row["trans_date_trans_time"])
            mask = (times < t) & (times >= t - np.timedelta64(24, "h"))
            expected = int(raw_sub.loc[mask, "is_high_amt"].sum())
            actual = int(row["recent_24h_high_amt_count"])
            assert expected == actual, (
                f"[FAIL] recent_24h_high_amt_count 불일치 cc_num={cc} trans_num={row['trans_num']} "
                f"expected={expected} actual={actual}"
            )
    print("[OK] recent_24h_high_amt_count 재계산 검증 통과 (표본 고객 각 20건)")


def verify_prior_normal_median(train_raw: pd.DataFrame, train_result: pd.DataFrame, n_sample: int = 5) -> None:
    rng = np.random.default_rng(7)
    cards = train_result["cc_num"].unique()
    sample_cards = rng.choice(cards, size=min(n_sample, len(cards)), replace=False)
    train_raw = train_raw.copy()
    train_raw["trans_date_trans_time"] = pd.to_datetime(train_raw["trans_date_trans_time"])

    for cc in sample_cards:
        sub = train_result.loc[train_result["cc_num"] == cc].sort_values("trans_date_trans_time")
        raw_sub = train_raw.loc[train_raw["cc_num"] == cc].sort_values("trans_date_trans_time")
        for _, row in sub.tail(10).iterrows():
            t = row["trans_date_trans_time"]
            prior_normal = raw_sub.loc[
                (raw_sub["trans_date_trans_time"] < t) & (raw_sub["is_fraud"] == 0),
                "amt",
            ]
            if len(prior_normal) == 0:
                assert pd.isna(row["prior_normal_median_amt"])
                continue
            expected = float(prior_normal.median())
            actual = row["prior_normal_median_amt"]
            assert np.isclose(expected, actual, atol=1e-6), (
                f"[FAIL] prior_normal_median_amt 불일치 cc_num={cc} trans_num={row['trans_num']} "
                f"expected={expected} actual={actual}"
            )
    print("[OK] prior_normal_median_amt 재계산 검증 통과 (미래/자기자신 정보 미사용 확인)")


def verify_no_future_leak_basic(train_df: pd.DataFrame, test_df: pd.DataFrame) -> None:
    first_train = train_df.sort_values("trans_date_trans_time").groupby("cc_num").head(1)
    assert (first_train["amt_zscore_card"] == 0).all(), "[FAIL] 카드별 첫 거래의 amt_zscore_card가 0이 아님"
    print("[OK] 카드별 '첫 거래' amt_zscore_card == 0 확인 (미래/전체이력 사용 아님)")

    assert test_df["is_fraud"].isin([0, 1]).all()
    print("[OK] 컬럼 스키마 및 is_fraud 값 범위 확인")


def run_verification(combined_raw: pd.DataFrame, train_raw: pd.DataFrame, train_df: pd.DataFrame, test_df: pd.DataFrame) -> None:
    print("\n=== 검증 시작 ===")
    verify_no_future_leak_basic(train_df, test_df)
    verify_recent_24h(combined_raw, pd.concat([train_df, test_df], ignore_index=True))
    verify_prior_normal_median(train_raw, train_df)
    print("=== 검증 끝: 위에 [FAIL]이 없으면 5개 변수 모두 시점 인과성 통과 ===\n")


## 6. 실행

In [21]:
train_path = find_path(CANDIDATE_TRAIN_PATHS)
test_path = find_path(CANDIDATE_TEST_PATHS)
print(f"train 원본: {train_path}")
print(f"test  원본: {test_path}")

train_raw_for_verify = load_raw(train_path, "train")
combined_raw_for_verify = pd.concat(
    [train_raw_for_verify, load_raw(test_path, "test")], ignore_index=True
)


train 원본: C:\Users\splen\OneDrive\Desktop\BDAI_\BOOSTMAP\Fraud-FDS-Project\data\raw\fraudTrain.csv
test  원본: C:\Users\splen\OneDrive\Desktop\BDAI_\BOOSTMAP\Fraud-FDS-Project\data\raw\fraudTest.csv


In [22]:
train_df, test_df = build_datasets(train_path, test_path)

print(f"train_11features: {train_df.shape}")
print(f"test_11features : {test_df.shape}")
train_df.head()


train_11features: (1296675, 15)
test_11features : (555719, 15)


,trans_num,trans_date_trans_time,cc_num,category,amt,trans_hour,age,recent_24h_high_amt_count,amt_to_prior_median_ratio,rolling_sum_amt_1h,amt_zscore_card,prior_normal_median_amt,count_30min,high_speed,is_fraud
0,0b242abb623afc578575680df30655b9,2019-01-01 00:00:18,2703186189652095,misc_net,4.97,0,30,0,NaN,4.97,0.0,NaN,0,0,0
1,1f76529f8574734946361c461b024d99,2019-01-01 00:00:44,630423337322,grocery_pos,107.23,0,40,0,NaN,107.23,0.0,NaN,1,0,0
2,a1a22d70485983eac12b5b88dad1cf95,2019-01-01 00:00:51,38859492057661,entertainment,220.11,0,56,0,NaN,220.11,0.0,NaN,1,0,0
3,6b849c168bdad6f867558c3793159a81,2019-01-01 00:01:16,3534093764340240,gas_transport,45.00,0,51,0,NaN,45.00,0.0,NaN,1,0,0
4,a41d7549acf90789359a9aa5346dcb46,2019-01-01 00:03:06,375534208663984,misc_pos,41.96,0,32,0,NaN,41.96,0.0,NaN,1,0,0


In [23]:
run_verification(combined_raw_for_verify, train_raw_for_verify, train_df, test_df)



=== 검증 시작 ===
[OK] 카드별 '첫 거래' amt_zscore_card == 0 확인 (미래/전체이력 사용 아님)
[OK] 컬럼 스키마 및 is_fraud 값 범위 확인
[OK] recent_24h_high_amt_count 재계산 검증 통과 (표본 고객 각 20건)
[OK] prior_normal_median_amt 재계산 검증 통과 (미래/자기자신 정보 미사용 확인)
=== 검증 끝: 위에 [FAIL]이 없으면 5개 변수 모두 시점 인과성 통과 ===



In [25]:
train_df.to_csv(OUT_TRAIN, index=False, encoding="utf-8-sig")
test_df.to_csv(OUT_TEST, index=False, encoding="utf-8-sig")
print(f"저장 완료: {OUT_TRAIN.resolve()}")
print(f"저장 완료: {OUT_TEST.resolve()}")


저장 완료: C:\Users\splen\OneDrive\Desktop\BDAI_\BOOSTMAP\Fraud-FDS-Project\revision\train_11features.csv
저장 완료: C:\Users\splen\OneDrive\Desktop\BDAI_\BOOSTMAP\Fraud-FDS-Project\revision\test_11features.csv
